# KPI Aziendali e Analisi Flotta – Buby Rent

**Obiettivo:** calcolare in modo riproducibile gli 8 KPI definiti in `dai/README.md` §5 ed esportare le tabelle pronte per la dashboard WSA e il report finale in `../../kpi/`:
1. **Tasso di utilizzo della flotta** (giorni noleggiati / giorni flotta, per sede, tipo veicolo e mese).
2. **Giorni di fermo** per manutenzioni e danni (per sede e tipo intervento).
3. **Costi di manutenzione** per veicolo e per km percorso.
4. **Ricavi** per sede e mese, ricavo medio per noleggio e per veicolo.
5. **Tasso di annullamento e di no-show** per sede e canale.
6. **Flussi one-way e sbilanciamento flotta** (entrate − uscite per sede).
7. **Valutazione media dei clienti** (per sede, tipo veicolo, presenza danni).
8. **Segnalazioni per categoria e urgenza** (anteprima sul test set della baseline `05`; verrà estesa all'intero dataset dopo la ratifica SAL 1).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

BASE_FILES = Path("../../files")
BASE_PULITI = Path("../../dati_puliti")
BASE_MODELLI = Path("../../modelli")
OUT_KPI = Path("../../kpi")
OUT_KPI.mkdir(parents=True, exist_ok=True)

nol = pd.read_csv(BASE_PULITI / "noleggi_tutti_pulito.csv")
man = pd.read_csv(BASE_PULITI / "manutenzioni_tutti_pulito.csv")
flotta = pd.read_csv(BASE_FILES / "flotta.csv")
sedi_df = pd.read_csv(BASE_FILES / "sedi.csv")

nome_sede = sedi_df.set_index("id_sede")["nome_sede"].str.replace(r"^Buby Rent ", "", regex=True).to_dict()
flotta["tipo_veicolo"] = flotta["tipo_mezzo"].replace({"e-bike": "ebike"})
veicolo_sede = flotta.set_index("id_veicolo")["id_sede"].to_dict()
veicolo_tipo = flotta.set_index("id_veicolo")["tipo_veicolo"].to_dict()

# Finestra osservata del dataset: 2025-11-01 .. 2026-11-10 (375 giorni)
DATA_INIZIO = pd.Timestamp("2025-11-01")
DATA_FINE = pd.Timestamp("2026-11-10")
GIORNI_PERIODO = (DATA_FINE - DATA_INIZIO).days + 1

nol["dt_ritiro"] = pd.to_datetime(nol["data_ritiro"])
nol["dt_fine"] = pd.to_datetime(nol["data_riconsegna_effettiva"].fillna(nol["data_riconsegna_prevista"]))
nol["mese"] = nol["dt_ritiro"].dt.to_period("M").astype(str)

man["dt_ingresso"] = pd.to_datetime(man["data_ingresso"])
man["mese"] = man["dt_ingresso"].dt.to_period("M").astype(str)
man["id_sede"] = man["id_veicolo"].map(veicolo_sede)
man["tipo_veicolo"] = man["id_veicolo"].map(veicolo_tipo)

nol_eff = nol[nol["stato_noleggio"] != "annullata"].copy()
nol_conc = nol[nol["stato_noleggio"] == "conclusa"].copy()

# Giorni osservati per ciascun mese nella finestra [DATA_INIZIO, DATA_FINE]
calendario = pd.DataFrame({"data": pd.date_range(DATA_INIZIO, DATA_FINE, freq="D")})
calendario["mese"] = calendario["data"].dt.to_period("M").astype(str)
giorni_per_mese = calendario.groupby("mese").size().to_dict()

print(
    f"Periodo: {DATA_INIZIO.date()} -> {DATA_FINE.date()} ({GIORNI_PERIODO} giorni) | "
    f"Flotta: {len(flotta)} veicoli | Noleggi: {len(nol)} ({len(nol_conc)} conclusi, {len(nol_eff)} effettivi) | "
    f"Manutenzioni: {len(man)} ({man['id_sede'].notna().sum()} associate a flotta)."
)

Periodo: 2025-11-01 -> 2026-11-10 (375 giorni) | Flotta: 64 veicoli | Noleggi: 4997 (4545 conclusi, 4567 effettivi) | Manutenzioni: 861 (859 associate a flotta).


## 1. Tasso di utilizzo flotta, ricavi, annullamenti e no-show (`kpi_sede_mese.csv`)

Per ogni `(sede, mese)` confrontiamo i giorni effettivamente noleggiati (`durata_giorni` dei noleggi non annullati) con la capacità della flotta (`n_veicoli × giorni_osservati_mese`), tenendo conto dei giorni di fermo manutenzione (`tasso_utilizzo_lordo_%` e `tasso_utilizzo_netto_%` sui soli giorni non in officina).

In [2]:
# Capacità flotta per sede e per (sede, tipo)
flotta_sede = flotta.groupby("id_sede").size().to_dict()
flotta_sede_tipo = flotta.groupby(["id_sede", "tipo_veicolo"]).size().to_dict()

SEDI = sorted(sedi_df["id_sede"].unique())
MESI = sorted(giorni_per_mese)
griglia_sm = pd.MultiIndex.from_product([SEDI, MESI], names=["id_sede", "mese"]).to_frame(index=False)

# Aggregati mensili su noleggi
agg_all = nol.groupby(["sede_ritiro", "mese"]).agg(
    noleggi_totali=("id_noleggio", "size"),
    annullati=("stato_noleggio", lambda s: (s == "annullata").sum()),
    no_show=("no_show", lambda s: (s == "si").sum()),
).reset_index().rename(columns={"sede_ritiro": "id_sede"})

agg_eff = nol_eff.groupby(["sede_ritiro", "mese"]).agg(
    noleggi_effettivi=("id_noleggio", "size"),
    giorni_noleggiati=("durata_giorni", "sum"),
    km_percorsi=("km_percorsi", "sum"),
    valutazione_media=("valutazione_cliente", "mean"),
).reset_index().rename(columns={"sede_ritiro": "id_sede"})

agg_conc = nol_conc.groupby(["sede_ritiro", "mese"]).agg(
    ricavi_eur=("importo_totale_eur", "sum"),
    ricavo_medio_eur=("importo_totale_eur", "mean"),
).reset_index().rename(columns={"sede_ritiro": "id_sede"})

agg_man_m = man.dropna(subset=["id_sede"]).groupby(["id_sede", "mese"]).agg(
    interventi_manut=("id_intervento", "size"),
    giorni_fermo=("giorni_fermo", "sum"),
    costo_manut_eur=("costo_eur", "sum"),
).reset_index()

sm = (
    griglia_sm
    .merge(agg_all, on=["id_sede", "mese"], how="left")
    .merge(agg_eff, on=["id_sede", "mese"], how="left")
    .merge(agg_conc, on=["id_sede", "mese"], how="left")
    .merge(agg_man_m, on=["id_sede", "mese"], how="left")
)

for col in ["noleggi_totali", "annullati", "no_show", "noleggi_effettivi", "giorni_noleggiati", "km_percorsi", "ricavi_eur", "interventi_manut", "giorni_fermo", "costo_manut_eur"]:
    sm[col] = sm[col].fillna(0)

sm["sede"] = sm["id_sede"].map(nome_sede)
sm["n_veicoli"] = sm["id_sede"].map(flotta_sede)
sm["giorni_flotta"] = sm["n_veicoli"] * sm["mese"].map(giorni_per_mese)
sm["giorni_disponibili_netti"] = np.maximum(sm["giorni_flotta"] - sm["giorni_fermo"], 1)
sm["tasso_utilizzo_%"] = (sm["giorni_noleggiati"] / sm["giorni_flotta"] * 100).round(2)
sm["tasso_utilizzo_netto_%"] = (sm["giorni_noleggiati"] / sm["giorni_disponibili_netti"] * 100).round(2)
sm["tasso_annullamento_%"] = np.where(sm["noleggi_totali"] > 0, sm["annullati"] / sm["noleggi_totali"] * 100, 0).round(2)
sm["tasso_no_show_%"] = np.where(sm["noleggi_totali"] > 0, sm["no_show"] / sm["noleggi_totali"] * 100, 0).round(2)
sm["ricavi_eur"] = sm["ricavi_eur"].round(2)
sm["ricavo_medio_eur"] = sm["ricavo_medio_eur"].round(2)
sm["costo_manut_eur"] = sm["costo_manut_eur"].round(2)
sm["valutazione_media"] = sm["valutazione_media"].round(2)

cols_sm = [
    "id_sede", "sede", "mese", "n_veicoli", "noleggi_totali", "noleggi_effettivi",
    "ricavi_eur", "ricavo_medio_eur", "giorni_noleggiati", "giorni_flotta", "giorni_fermo",
    "tasso_utilizzo_%", "tasso_utilizzo_netto_%", "tasso_annullamento_%", "tasso_no_show_%",
    "interventi_manut", "costo_manut_eur", "valutazione_media",
]
SM_OUT = OUT_KPI / "kpi_sede_mese.csv"
sm[cols_sm].to_csv(SM_OUT, index=False)

# Utilizzo per sede x tipo veicolo sull'intero periodo
ut_st = (
    nol_eff.groupby(["sede_ritiro", "tipologia_mezzo"])["durata_giorni"].sum()
    .rename("giorni_noleggiati").reset_index()
    .rename(columns={"sede_ritiro": "id_sede", "tipologia_mezzo": "tipo_veicolo"})
)
ut_st["n_veicoli"] = [flotta_sede_tipo[(s, t)] for s, t in zip(ut_st["id_sede"], ut_st["tipo_veicolo"])]
ut_st["tasso_utilizzo_%"] = (ut_st["giorni_noleggiati"] / (ut_st["n_veicoli"] * GIORNI_PERIODO) * 100).round(1)
print(f"Salvato {SM_OUT} ({len(sm)} righe). Tasso di utilizzo medio (%) per sede × tipo veicolo:")
ut_st.pivot(index="id_sede", columns="tipo_veicolo", values="tasso_utilizzo_%")

Salvato ..\..\kpi\kpi_sede_mese.csv (65 righe). Tasso di utilizzo medio (%) per sede × tipo veicolo:


tipo_veicolo,auto,ebike,furgone
id_sede,,,
ALB,71.4,26.1,55.9
BAR,73.5,18.6,51.5
CAS,69.8,22.7,58.0
STR,69.7,27.9,43.7
TOR,71.4,25.7,57.1


## 2. Manutenzioni, costi per veicolo/km, flussi one-way e scorecard sedi

Calcoliamo:
- dettaglio manutenzioni per sede e tipo intervento (`kpi_manutenzioni_tipo.csv`);
- matrice origine → destinazione dei noleggi one-way (riconsegna in altra sede) e saldo netto per sede;
- scorecard riassuntiva per sede + totale azienda (`kpi_riepilogo_sedi.csv`).

In [3]:
# 1) Manutenzioni per sede e tipo intervento
man_s = man.dropna(subset=["id_sede"]).copy()
man_s["sede"] = man_s["id_sede"].map(nome_sede)
man_tipo = (
    man_s.groupby(["id_sede", "sede", "tipo_intervento"])
    .agg(
        interventi=("id_intervento", "size"),
        giorni_fermo=("giorni_fermo", "sum"),
        fermo_medio_gg=("giorni_fermo", "mean"),
        costo_totale_eur=("costo_eur", "sum"),
        costo_medio_eur=("costo_eur", "mean"),
        pct_addebitato_cliente=("addebitato_cliente", lambda s: (s == "si").mean() * 100),
    )
    .reset_index()
)
for c in ["giorni_fermo", "fermo_medio_gg", "costo_totale_eur", "costo_medio_eur", "pct_addebitato_cliente"]:
    man_tipo[c] = man_tipo[c].round(2)

MAN_OUT = OUT_KPI / "kpi_manutenzioni_tipo.csv"
man_tipo.to_csv(MAN_OUT, index=False)

# 2) Flussi one-way (su noleggi effettivi)
ow = nol_eff[nol_eff["sede_ritiro"] != nol_eff["sede_riconsegna"]].copy()
matrice_ow = pd.crosstab(ow["sede_ritiro"], ow["sede_riconsegna"])
uscite_ow = ow.groupby("sede_ritiro").size().reindex(SEDI, fill_value=0)
entrate_ow = ow.groupby("sede_riconsegna").size().reindex(SEDI, fill_value=0)
saldo_ow = (entrate_ow - uscite_ow).to_dict()

# 3) Scorecard riassuntiva per sede + TOTALE
rows_sc = []
for sid in SEDI:
    n_all = nol[nol["sede_ritiro"] == sid]
    n_ef = nol_eff[nol_eff["sede_ritiro"] == sid]
    n_co = nol_conc[nol_conc["sede_ritiro"] == sid]
    m_s = man_s[man_s["id_sede"] == sid]
    nv = flotta_sede[sid]
    g_flotta = nv * GIORNI_PERIODO
    g_nol = float(n_ef["durata_giorni"].sum())
    g_fermo = float(m_s["giorni_fermo"].sum())
    km_tot = float(n_ef["km_percorsi"].sum())
    costo_m = float(m_s["costo_eur"].sum())
    ricavi = float(n_co["importo_totale_eur"].sum())
    rows_sc.append({
        "id_sede": sid,
        "sede": nome_sede[sid],
        "n_veicoli": nv,
        "noleggi_totali": len(n_all),
        "noleggi_effettivi": len(n_ef),
        "tasso_utilizzo_%": round(g_nol / g_flotta * 100, 1),
        "giorni_fermo": int(round(g_fermo)),
        "fermo_flotta_%": round(g_fermo / g_flotta * 100, 1),
        "ricavi_eur": round(ricavi, 2),
        "ricavo_medio_eur": round(ricavi / len(n_co), 2),
        "ricavo_per_veicolo_eur": round(ricavi / nv, 2),
        "costo_manut_eur": round(costo_m, 2),
        "costo_manut_per_veicolo_eur": round(costo_m / nv, 2),
        "costo_manut_per_100km_eur": round(costo_m / km_tot * 100, 2) if km_tot > 0 else np.nan,
        "tasso_annullamento_%": round((n_all["stato_noleggio"] == "annullata").mean() * 100, 1),
        "tasso_no_show_%": round((n_all["no_show"] == "si").mean() * 100, 1),
        "saldo_one_way": int(saldo_ow[sid]),
        "valutazione_media": round(float(n_all["valutazione_cliente"].mean()), 2),
    })

# Riga TOTALE aziendale (include anche i 2 interventi manutenzione con id_veicolo irrisolto)
g_flotta_tot = len(flotta) * GIORNI_PERIODO
g_nol_tot = float(nol_eff["durata_giorni"].sum())
g_fermo_tot = float(man["giorni_fermo"].sum())
km_tot_all = float(nol_eff["km_percorsi"].sum())
costo_m_tot = float(man["costo_eur"].sum())
ricavi_tot = float(nol_conc["importo_totale_eur"].sum())
rows_sc.append({
    "id_sede": "TOTALE",
    "sede": "Tutte le sedi",
    "n_veicoli": len(flotta),
    "noleggi_totali": len(nol),
    "noleggi_effettivi": len(nol_eff),
    "tasso_utilizzo_%": round(g_nol_tot / g_flotta_tot * 100, 1),
    "giorni_fermo": int(round(g_fermo_tot)),
    "fermo_flotta_%": round(g_fermo_tot / g_flotta_tot * 100, 1),
    "ricavi_eur": round(ricavi_tot, 2),
    "ricavo_medio_eur": round(ricavi_tot / len(nol_conc), 2),
    "ricavo_per_veicolo_eur": round(ricavi_tot / len(flotta), 2),
    "costo_manut_eur": round(costo_m_tot, 2),
    "costo_manut_per_veicolo_eur": round(costo_m_tot / len(flotta), 2),
    "costo_manut_per_100km_eur": round(costo_m_tot / km_tot_all * 100, 2),
    "tasso_annullamento_%": round((nol["stato_noleggio"] == "annullata").mean() * 100, 1),
    "tasso_no_show_%": round((nol["no_show"] == "si").mean() * 100, 1),
    "saldo_one_way": 0,
    "valutazione_media": round(float(nol["valutazione_cliente"].mean()), 2),
})

scorecard = pd.DataFrame(rows_sc)
SC_OUT = OUT_KPI / "kpi_riepilogo_sedi.csv"
scorecard.to_csv(SC_OUT, index=False)

print(f"Salvati {MAN_OUT} ({len(man_tipo)} righe) e {SC_OUT} ({len(scorecard)} righe).")
print(f"\nOne-way effettivi: {len(ow)} | Saldo netto (entrate - uscite): {saldo_ow}")
display(matrice_ow)
scorecard

Salvati ..\..\kpi\kpi_manutenzioni_tipo.csv (55 righe) e ..\..\kpi\kpi_riepilogo_sedi.csv (6 righe).

One-way effettivi: 107 | Saldo netto (entrate - uscite): {'ALB': 10, 'BAR': 2, 'CAS': -4, 'STR': 5, 'TOR': -13}


sede_riconsegna,ALB,BAR,CAS,STR,TOR
sede_ritiro,,,,,
ALB,0,6,2,3,4
BAR,4,0,2,8,7
CAS,6,4,0,4,6
STR,6,6,5,0,2
TOR,9,7,7,9,0


,id_sede,sede,n_veicoli,noleggi_totali,noleggi_effettivi,tasso_utilizzo_%,giorni_fermo,fermo_flotta_%,ricavi_eur,ricavo_medio_eur,ricavo_per_veicolo_eur,costo_manut_eur,costo_manut_per_veicolo_eur,costo_manut_per_100km_eur,tasso_annullamento_%,tasso_no_show_%,saldo_one_way,valutazione_media
0,ALB,Alba,11,857,792,56.2,716,17.4,129147.16,164.10,11740.65,55869.8,5079.07,21.52,7.6,0.8,10,4.28
1,BAR,Bardonecchia,10,677,616,54.8,474,12.6,173381.81,283.30,17338.18,33169.2,3316.92,18.80,9.0,2.2,2,4.18
2,CAS,Casale Monferrato,10,811,741,53.3,607,16.2,110257.94,149.60,11025.79,49885.5,4988.55,26.45,8.6,1.6,-4,4.20
3,STR,Stresa,11,921,838,52.1,608,14.7,120138.30,143.88,10921.66,48871.1,4442.83,21.74,9.0,1.4,5,4.17
4,TOR,Torino Lingotto,22,1731,1580,58.4,1478,17.9,302417.12,192.13,13746.23,98315.4,4468.88,24.23,8.7,1.8,-13,4.24
5,TOTALE,Tutte le sedi,64,4997,4567,55.6,3897,16.2,835342.33,183.79,13052.22,286848.1,4482.00,22.85,8.6,1.6,0,4.22


## 3. Approfondimenti (valutazione clienti, canali, segnalazioni da `05`) e verifiche finali

In [4]:
print("--- Tasso annullamento e no-show per canale prenotazione ---")
can_kpi = nol.groupby("canale_prenotazione").agg(
    noleggi=("id_noleggio", "size"),
    annullamento_pct=("stato_noleggio", lambda s: round((s == "annullata").mean() * 100, 1)),
    no_show_pct=("no_show", lambda s: round((s == "si").mean() * 100, 1)),
)
display(can_kpi)

print("--- Valutazione media per tipo veicolo e presenza danni ---")
print(nol.groupby("tipologia_mezzo")["valutazione_cliente"].agg(["count", "mean"]).round(2))
print(nol.groupby("danni_segnalati")["valutazione_cliente"].agg(["count", "mean"]).round(2))

# KPI #8: distribuzione segnalazioni predette dalla baseline (test set del notebook 05)
base_nlp = BASE_MODELLI / "predizioni_baseline_test.csv"
if base_nlp.is_file():
    nlp_df = pd.read_csv(base_nlp)
    print(f"\n--- KPI #8 (anteprima su test set baseline 05, n={len(nlp_df)}) ---")
    display(pd.crosstab(nlp_df["cat_app_base"], nlp_df["urg_base"], margins=True))

# Verifiche finali di coerenza
assert SM_OUT.is_file() and MAN_OUT.is_file() and SC_OUT.is_file()
assert len(sm) == len(SEDI) * len(MESI)
assert (sm["tasso_utilizzo_%"].between(0, 100)).all(), "Utilizzo mensile fuori scala [0, 100]"
assert scorecard.loc[scorecard["id_sede"] != "TOTALE", "saldo_one_way"].sum() == 0, "Saldo one-way deve sommare a 0"
assert abs(scorecard.loc[scorecard["id_sede"] == "TOTALE", "ricavi_eur"].iat[0] - 835342.33) < 0.05
print("Tutte le verifiche KPI superate.")

--- Tasso annullamento e no-show per canale prenotazione ---


,noleggi,annullamento_pct,no_show_pct
canale_prenotazione,,,
app,1103,9.2,2.0
portale partner,518,9.7,1.4
sportello,784,8.3,1.3
telefono,724,7.2,1.5
web,1868,8.6,1.6


--- Valutazione media per tipo veicolo e presenza danni ---
                 count  mean
tipologia_mezzo             
auto               992  4.18
ebike              753  4.25
furgone            285  4.27
                 count  mean
danni_segnalati             
no                1573  4.27
si                 158  3.77

--- KPI #8 (anteprima su test set baseline 05, n=437) ---


urg_base,alta,bassa,media,All
cat_app_base,,,,
altro,2,253,25,280
danno,26,50,39,115
guasto,1,28,13,42
All,29,331,77,437


Tutte le verifiche KPI superate.
